# 🔍 RAG — Retrieval-Augmented Generation, do Zero ao Avançado

### Como dar memória e conhecimento factual a um LLM sem re-treiná-lo

Este notebook implementa um **pipeline de RAG completo** do zero, sem frameworks
como LangChain — para que você entenda cada peça antes de usar abstrações.

**Roteiro:**

| Parte | Conteúdo |
|-------|----------|
| **1** | Intuição: por que RAG? Fine-tuning vs RAG vs prompting |
| **2** | O pipeline RAG ponta a ponta |
| **3** | Preparação de documentos e chunking |
| **4** | Embeddings — transformando texto em vetores |
| **5** | Vector Store — busca por similaridade |
| **6** | Retrieval — encontrando os chunks relevantes |
| **7** | Generation — gerando a resposta com contexto |
| **8** | Pipeline completo integrado |
| **9** | Experimentos: chunk size, overlap, top-k, com vs sem RAG |
| **10** | Técnicas avançadas: reranking, HyDE, avaliação |

**Modelo:** `Qwen/Qwen2.5-0.5B-Instruct` (geração) + `all-MiniLM-L6-v2` (embeddings)

**Base de conhecimento:** Documento sobre as regras e a história do xadrez
(criado sinteticamente — para simular um domínio que o modelo pode não dominar).

> 💡 GPU T4 recomendada mas não obrigatória — o pipeline de RAG roda em CPU também.

---
## 1. 🧠 Intuição: Por Que RAG?

### O problema fundamental dos LLMs

LLMs são treinados em dados até uma **data de corte**. Depois disso:
- Não sabem de eventos recentes
- Não conhecem seus documentos internos
- Podem "alucinar" fatos com confiança

### Três formas de dar conhecimento a um LLM

```
1. PROMPTING (in-context learning)
   "Baseado neste texto: [texto completo], responda: ..."
   ✅ Simples
   ❌ Limitado pelo tamanho da context window
   ❌ Não escala para muitos documentos

2. FINE-TUNING
   Treinar o modelo nos seus dados
   ✅ Modelo internaliza o conhecimento
   ❌ Caro e lento para atualizar
   ❌ Não confiável para fatos específicos (alucinação)
   ❌ Precisa re-treinar quando dados mudam

3. RAG (Retrieval-Augmented Generation)    ← FOCO DESTE NOTEBOOK
   Buscar documentos relevantes e incluí-los no prompt
   ✅ Conhecimento sempre atualizado (basta atualizar os docs)
   ✅ Rastreável (você sabe de onde veio a informação)
   ✅ Sem re-treinamento
   ✅ Escala para milhões de documentos
   ⚠️ Qualidade depende da busca (garbage retrieval → garbage answer)
```

### Quando usar cada abordagem?

| Necessidade | Melhor abordagem | Exemplo |
|---|---|---|
| Ajustar estilo/formato | Fine-tuning | "Responda sempre em bullet points" |
| Conhecimento factual atualizado | **RAG** | "O que diz a ata da reunião de ontem?" |
| Domínio especializado + fatos | **RAG + Fine-tuning** | Assistente médico com acesso a prontuários |
| Tarefa genérica | Prompting | "Traduza este texto" |

---
## 2. 🔄 O Pipeline RAG Ponta a Ponta

```
══════════════════ INDEXAÇÃO (offline, uma vez) ══════════════════

  Documentos                      Embeddings            Vector Store
  ┌─────────┐     Chunking       ┌──────────┐          ┌──────────┐
  │ Doc 1   │  ──────────────►   │ Chunk 1  │ ──emb──► │ [0.2, …] │
  │ Doc 2   │   Dividir em       │ Chunk 2  │ ──emb──► │ [0.8, …] │
  │ Doc 3   │   pedaços          │ Chunk 3  │ ──emb──► │ [0.1, …] │
  │   ...   │   menores          │   ...    │          │   ...    │
  └─────────┘                    └──────────┘          └──────────┘

══════════════════ CONSULTA (online, por pergunta) ══════════════

  Pergunta         Embedding        Busca           Contexto
  ┌──────────┐     ┌──────┐     ┌──────────┐     ┌──────────────┐
  │ "Como o  │──►  │[0.3, │──►  │ Top-K    │──►  │ Chunk 2 (✓)  │
  │  bispo   │     │ …]   │     │ similar  │     │ Chunk 7 (✓)  │
  │  move?" │     └──────┘     └──────────┘     │ Chunk 3 (✓)  │
  └──────────┘                                    └──────────────┘
                                                        │
                                                        ▼
                                               ┌────────────────┐
  Prompt Final:                                │     LLM        │
  "Baseado nos seguintes trechos:              │  ┌──────────┐  │
   [Chunk 2] [Chunk 7] [Chunk 3]              │  │ Qwen 0.5B│  │
   Responda: Como o bispo move?"    ──────►    │  └──────────┘  │
                                               │                │
                                               │  "O bispo move │
                                               │   na diagonal" │
                                               └────────────────┘
```

### Os 5 componentes:

1. **Chunking:** dividir documentos em pedaços menores
2. **Embedding:** transformar texto em vetores numéricos
3. **Vector Store:** armazenar e indexar os vetores
4. **Retrieval:** buscar os chunks mais relevantes para a pergunta
5. **Generation:** montar o prompt com contexto e gerar a resposta

---
## 3. 📦 Setup e Preparação de Documentos

In [ ]:
# !pip install -q sentence-transformers faiss-cpu transformers accelerate

import torch
import numpy as np
import matplotlib.pyplot as plt
import time
import re
import textwrap
from dataclasses import dataclass, field
from typing import List, Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# === Base de Conhecimento: Documento sobre Xadrez ===
# Um documento longo o suficiente para demonstrar chunking e retrieval,
# sobre um domínio onde podemos verificar se as respostas estão corretas.

CHESS_DOCUMENT = """
# Guia Completo de Xadrez

## História do Xadrez

O xadrez é um dos jogos mais antigos da humanidade. Sua origem remonta ao século VI na Índia, onde era conhecido como Chaturanga. O jogo se espalhou pela Pérsia, onde recebeu o nome de Shatranj, e posteriormente foi levado pelos árabes à Europa durante a Idade Média. As regras modernas foram estabelecidas por volta do século XV na Europa, quando a rainha e o bispo ganharam seus movimentos atuais de longo alcance.

O primeiro campeonato mundial oficial de xadrez foi disputado em 1886 entre Wilhelm Steinitz e Johannes Zukertort. Steinitz venceu e se tornou o primeiro campeão mundial oficial. Desde então, o título já foi detido por grandes nomes como Emanuel Lasker, José Raúl Capablanca, Alexander Alekhine, Mikhail Botvinnik, Bobby Fischer, Garry Kasparov e Magnus Carlsen.

Em 1997 ocorreu um marco histórico quando o computador Deep Blue, da IBM, derrotou Garry Kasparov em uma partida oficial. Este evento marcou o início da era da supremacia computacional no xadrez. Atualmente, os programas de xadrez baseados em redes neurais, como o Stockfish com NNUE e o AlphaZero do Google DeepMind, são significativamente mais fortes que qualquer jogador humano.

## O Tabuleiro

O tabuleiro de xadrez é composto por 64 casas dispostas em uma grade de 8 colunas por 8 fileiras. As casas alternam entre cores claras e escuras. As colunas são identificadas pelas letras de 'a' a 'h' da esquerda para a direita (do ponto de vista das peças brancas), e as fileiras pelos números de 1 a 8 de baixo para cima. Cada casa é identificada de forma única pela combinação da coluna e fileira, por exemplo 'e4' ou 'd7'.

O tabuleiro deve ser posicionado de forma que cada jogador tenha uma casa branca no canto inferior direito. As peças brancas são tradicionalmente posicionadas nas fileiras 1 e 2, e as peças pretas nas fileiras 7 e 8.

## As Peças e Seus Movimentos

### O Rei

O rei é a peça mais importante do jogo. Se o rei for capturado (xeque-mate), o jogo termina. O rei move-se exatamente uma casa em qualquer direção: horizontal, vertical ou diagonal. Apesar de ser a peça mais importante, é uma das mais lentas do tabuleiro. O rei não pode mover-se para uma casa que esteja atacada por uma peça adversária. O valor do rei é considerado infinito, pois sua perda significa a derrota.

### A Rainha (Dama)

A rainha é a peça mais poderosa do jogo. Ela combina os movimentos da torre e do bispo, podendo mover-se qualquer número de casas em linha reta nas direções horizontal, vertical e diagonal. A rainha não pode saltar sobre outras peças. Seu valor relativo é de aproximadamente 9 pontos. A rainha é frequentemente usada em ataques decisivos e é uma peça crucial no meio-jogo e no final.

### A Torre

A torre move-se qualquer número de casas na horizontal ou vertical, mas não pode saltar sobre outras peças. Cada jogador começa com duas torres, posicionadas nos cantos do tabuleiro (casas a1, h1 para as brancas e a8, h8 para as pretas). O valor relativo da torre é de aproximadamente 5 pontos. As torres são especialmente poderosas em colunas abertas (sem peões) e na sétima fileira, onde podem atacar peões adversários.

### O Bispo

O bispo move-se qualquer número de casas na diagonal. Cada jogador começa com dois bispos: um que se move apenas por casas claras e outro apenas por casas escuras. O bispo não pode saltar sobre outras peças. Seu valor relativo é de aproximadamente 3 pontos. O par de bispos (ter ambos os bispos) é considerado uma vantagem, pois juntos eles controlam casas de ambas as cores.

### O Cavalo

O cavalo tem o movimento mais singular do xadrez. Ele se move em forma de 'L': duas casas em uma direção (horizontal ou vertical) e depois uma casa perpendicularmente, ou vice-versa. O cavalo é a única peça que pode saltar sobre outras peças, tanto amigas quanto inimigas. Seu valor relativo é de aproximadamente 3 pontos. Cavalos são particularmente eficazes em posições fechadas onde outras peças têm mobilidade limitada.

### O Peão

O peão é a peça de menor valor (1 ponto) mas tem regras únicas. O peão avança uma casa para frente, mas captura na diagonal, uma casa à frente e para o lado. Em seu primeiro movimento, o peão pode avançar duas casas. Se um peão alcançar a última fileira do tabuleiro (fileira 8 para brancas, fileira 1 para pretas), ele deve ser promovido a qualquer outra peça, exceto rei — geralmente uma rainha. Cada jogador começa com oito peões.

## Movimentos Especiais

### Roque

O roque é um movimento especial envolvendo o rei e uma das torres. É o único movimento onde duas peças se movem ao mesmo tempo. No roque do lado do rei (roque curto), o rei move-se duas casas em direção à torre do lado do rei, e a torre salta para o outro lado do rei. No roque do lado da rainha (roque longo), o rei move-se duas casas em direção à torre do lado da rainha, e a torre salta para o outro lado do rei.

O roque só é permitido quando: o rei e a torre envolvida não se moveram anteriormente; não há peças entre o rei e a torre; o rei não está em xeque; o rei não passa por nenhuma casa atacada durante o roque; e o rei não termina em xeque.

### En Passant

O en passant é uma captura especial do peão. Quando um peão avança duas casas a partir de sua posição inicial e fica ao lado de um peão adversário, o peão adversário pode capturá-lo como se ele tivesse avançado apenas uma casa. Esta captura só pode ser feita imediatamente após o avanço de duas casas; se não for realizada no lance seguinte, o direito se perde.

### Promoção

Quando um peão alcança a última fileira do tabuleiro adversário, ele deve ser promovido a qualquer outra peça da mesma cor, exceto rei. Na grande maioria dos casos, o peão é promovido a rainha, por ser a peça mais poderosa. No entanto, em situações raras, pode ser vantajoso promover a cavalo para dar um xeque simultâneo, por exemplo.

## Conceitos Estratégicos Básicos

### Controle do Centro

As quatro casas centrais do tabuleiro (e4, d4, e5, d5) são consideradas as mais importantes estrategicamente. Peças posicionadas no centro controlam mais casas e têm mais mobilidade. As aberturas mais populares, como a Abertura Italiana, a Defesa Siciliana e o Gambito da Dama, todas buscam controlar o centro de diferentes maneiras.

### Desenvolvimento de Peças

Nos primeiros movimentos da partida (a abertura), é fundamental desenvolver as peças menores (cavalos e bispos) para casas ativas. Regras gerais incluem: desenvolver cavalos antes dos bispos; não mover a mesma peça duas vezes na abertura sem bom motivo; conectar as torres fazendo o roque; e evitar mover a rainha cedo demais, pois ela pode ser atacada e perder tempos.

### Estrutura de Peões

A estrutura de peões é a espinha dorsal de qualquer posição de xadrez. Peões não podem recuar, então cada avanço de peão é permanente e deve ser cuidadosamente considerado. Peões dobrados (dois peões da mesma cor na mesma coluna) são geralmente uma fraqueza. Peões isolados (sem peões da mesma cor em colunas adjacentes) também são vulneráveis. Peões passados (que não podem ser bloqueados por peões adversários) são uma grande vantagem no final.

### Segurança do Rei

Manter o rei seguro é uma prioridade ao longo de toda a partida. O roque é geralmente recomendado nas primeiras jogadas para colocar o rei em segurança atrás de uma barreira de peões. Um ataque ao rei adversário é frequentemente o objetivo principal no meio-jogo. Sinais de perigo incluem: rei no centro sem ter feito roque, peões avançados na frente do rei, e colunas ou diagonais abertas apontando para o rei.

## Fases da Partida

### Abertura

A abertura compreende aproximadamente os primeiros 10 a 15 movimentos. Os objetivos principais são: controlar o centro, desenvolver as peças, garantir a segurança do rei (geralmente através do roque), e conectar as torres. Existem centenas de aberturas catalogadas, desde as mais clássicas como a Ruy Lopez e a Defesa Francesa, até as mais modernas como a Defesa Caro-Kann e a Abertura Inglesa.

### Meio-Jogo

O meio-jogo começa após o desenvolvimento das peças e é a fase mais complexa. Nesta fase, os jogadores executam planos estratégicos e buscam vantagens táticas. Conceitos importantes incluem: ataques ao rei, sacrifícios de peças, controle de colunas abertas, domínio de casas fracas, e a coordenação entre as peças.

### Final

O final começa quando a maioria das peças foi trocada e os reis podem participar ativamente do jogo. No final, o rei torna-se uma peça ofensiva importante. Conceitos-chave incluem: a regra do quadrado (para calcular se um peão consegue promover), a oposição entre reis, e finais teóricos como rei e torre contra rei, ou rei e peão contra rei. O zugzwang, situação onde qualquer movimento piora a posição, é um conceito frequente nos finais.

## Sistema de Pontuação e Resultado

Uma partida de xadrez pode terminar de várias formas. O xeque-mate ocorre quando o rei está em xeque e não há nenhum movimento legal para escapar. O empate pode ocorrer por acordo mútuo, por afogamento (stalemate), quando o jogador não está em xeque mas não tem movimentos legais, por repetição de posição (a mesma posição ocorre três vezes), pela regra dos 50 movimentos (50 lances sem captura ou movimento de peão), ou por material insuficiente para dar xeque-mate.

Em torneios, a vitória vale 1 ponto, o empate vale 0.5 pontos para cada jogador, e a derrota vale 0 pontos. O sistema de classificação Elo, criado por Arpad Elo, é usado para medir a força relativa dos jogadores. Um jogador iniciante tem rating em torno de 800, um jogador intermediário cerca de 1500, um mestre nacional em torno de 2200, um grande mestre acima de 2500, e o campeão mundial tipicamente acima de 2800.
"""

print(f'Documento carregado: {len(CHESS_DOCUMENT):,} caracteres')
print(f'Parágrafos: {len([p for p in CHESS_DOCUMENT.split(chr(10)+chr(10)) if p.strip()])}')
print(f'\nPrimeiras 200 chars:')
print(CHESS_DOCUMENT.strip()[:200])

---
## 4. ✂️ Chunking — Dividindo Documentos em Pedaços

### Por que chunking?

Não podemos colocar o documento inteiro no prompt (pode exceder o contexto).
Também não queremos — queremos buscar **apenas os trechos relevantes**.

### Estratégias de chunking:

| Estratégia | Como funciona | Prós | Contras |
|---|---|---|---|
| **Fixed size** | Corta a cada N caracteres | Simples, previsível | Pode cortar no meio de frase |
| **Sentence** | Divide por sentenças | Semântica preservada | Chunks muito pequenos |
| **Paragraph** | Divide por parágrafos | Bom contexto | Tamanho irregular |
| **Recursive** | Tenta parágrafos, depois frases, depois caracteres | Melhor balanço | Mais complexo |
| **Semantic** | Agrupa por similaridade semântica | Chunks coerentes | Lento, precisa embeddings |

### O overlap

```
Sem overlap:   [Chunk 1         ] [Chunk 2         ] [Chunk 3         ]
Com overlap:   [Chunk 1         ]
                     [Chunk 2         ]
                           [Chunk 3         ]
```

Overlap garante que informação na fronteira entre chunks não se perca.

In [ ]:
# === Implementar múltiplas estratégias de chunking ===

@dataclass
class Chunk:
    """Um pedaço de texto com metadados."""
    text: str
    index: int
    source: str = 'chess_guide'
    metadata: dict = field(default_factory=dict)

    @property
    def char_count(self):
        return len(self.text)

    def __repr__(self):
        preview = self.text[:50].replace('\n', ' ')
        return f'Chunk({self.index}, {self.char_count} chars, "{preview}...")'


def chunk_fixed_size(text, chunk_size=500, overlap=50):
    """Chunking por tamanho fixo com overlap."""
    chunks = []
    start = 0
    idx = 0
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append(Chunk(text=chunk_text, index=idx,
                                metadata={'strategy': 'fixed', 'start': start, 'end': end}))
            idx += 1
        start += chunk_size - overlap
    return chunks


def chunk_by_paragraph(text, min_size=100, max_size=1000):
    """Chunking por parágrafo, agrupando parágrafos curtos."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    current = ''
    idx = 0

    for para in paragraphs:
        # Se adicionar este parágrafo excede max_size, salvar o atual
        if current and len(current) + len(para) > max_size:
            chunks.append(Chunk(text=current.strip(), index=idx,
                                metadata={'strategy': 'paragraph'}))
            idx += 1
            current = para
        else:
            current = current + '\n\n' + para if current else para

    # Último chunk
    if current.strip():
        chunks.append(Chunk(text=current.strip(), index=idx,
                            metadata={'strategy': 'paragraph'}))
    return chunks


def chunk_recursive(text, chunk_size=500, overlap=50):
    """Chunking recursivo: tenta parágrafos → sentenças → caracteres."""
    separators = ['\n\n', '\n', '. ', ' ']

    def _split(text, seps):
        if not seps or len(text) <= chunk_size:
            return [text]

        sep = seps[0]
        parts = text.split(sep)
        chunks_out = []
        current = ''

        for part in parts:
            candidate = current + sep + part if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks_out.append(current)
                # Se este part sozinho excede, recursão com próximo separador
                if len(part) > chunk_size:
                    chunks_out.extend(_split(part, seps[1:]))
                    current = ''
                else:
                    current = part

        if current:
            chunks_out.append(current)
        return chunks_out

    raw_chunks = _split(text, separators)
    return [Chunk(text=c.strip(), index=i, metadata={'strategy': 'recursive'})
            for i, c in enumerate(raw_chunks) if c.strip()]


print('✅ Estratégias de chunking implementadas: fixed_size, paragraph, recursive')

In [ ]:
# === 🔬 Experimento 1: Comparar estratégias de chunking ===

strategies = {
    'Fixed (500, overlap=50)': chunk_fixed_size(CHESS_DOCUMENT, 500, 50),
    'Fixed (300, overlap=50)': chunk_fixed_size(CHESS_DOCUMENT, 300, 50),
    'Paragraph': chunk_by_paragraph(CHESS_DOCUMENT),
    'Recursive (500)': chunk_recursive(CHESS_DOCUMENT, 500, 50),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, chunks) in zip(axes.flat, strategies.items()):
    sizes = [c.char_count for c in chunks]
    ax.bar(range(len(sizes)), sizes, color='steelblue', alpha=0.7)
    ax.set_title(f'{name}\n({len(chunks)} chunks, média={np.mean(sizes):.0f} chars)', fontweight='bold')
    ax.set_xlabel('Chunk index')
    ax.set_ylabel('Caracteres')
    ax.axhline(y=np.mean(sizes), color='red', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Comparação de Estratégias de Chunking', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Resumo
print(f'{"Estratégia":>30} | {"Chunks":>6} | {"Média":>6} | {"Min":>5} | {"Max":>5}')
print('-' * 70)
for name, chunks in strategies.items():
    sizes = [c.char_count for c in chunks]
    print(f'{name:>30} | {len(chunks):>6} | {np.mean(sizes):>6.0f} | {min(sizes):>5} | {max(sizes):>5}')

# Usar recursive para o restante
CHUNKS = chunk_recursive(CHESS_DOCUMENT, chunk_size=500, overlap=50)
print(f'\n✅ Usando chunking recursivo: {len(CHUNKS)} chunks')

In [ ]:
# Inspecionar alguns chunks
print('📋 Exemplos de chunks:\n')
for chunk in CHUNKS[:3]:
    text_preview = chunk.text[:150].replace('\n', ' ↵ ')
    print(f'  Chunk {chunk.index}: ({chunk.char_count} chars)')
    print(f'    "{text_preview}..."')
    print()

---
## 5. 🧲 Embeddings — Transformando Texto em Vetores

### O que são embeddings?

Embeddings mapeiam texto → vetor numérico, onde textos **semanticamente similares**
ficam **próximos** no espaço vetorial.

```
"O bispo move na diagonal"     → [0.82, -0.15, 0.33, ...]
"Como o bispo se movimenta?"   → [0.80, -0.12, 0.35, ...]  ← próximo!
"A receita de bolo de chocolate" → [-0.45, 0.91, 0.02, ...] ← distante!
```

### Modelos de embedding:

| Modelo | Dimensão | Velocidade | Qualidade |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Rápido | Boa |
| `bge-small-en-v1.5` | 384 | Rápido | Muito boa |
| `nomic-embed-text` | 768 | Médio | Excelente |
| `text-embedding-3-small` (OpenAI) | 1536 | API | Excelente |

Vamos usar `all-MiniLM-L6-v2` — é pequeno, rápido e funciona em CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

# Carregar modelo de embedding
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

print(f'Modelo de embedding: all-MiniLM-L6-v2')
print(f'Dimensão do vetor:   {EMBED_DIM}')
print(f'Max sequence length: {embed_model.max_seq_length}')

In [ ]:
# === 🔬 Experimento 2: Visualizando o espaço de embeddings ===
demo_sentences = [
    # Grupo 1: Movimentos de peças
    'O bispo move na diagonal',
    'A torre move em linha reta horizontal e vertical',
    'O cavalo move em forma de L',
    'A rainha move em qualquer direção',
    # Grupo 2: Conceitos estratégicos
    'Controlar o centro do tabuleiro é importante',
    'Desenvolva suas peças na abertura',
    'A segurança do rei é uma prioridade',
    # Grupo 3: Irrelevante
    'O preço do café subiu hoje',
    'Python é uma linguagem de programação',
    'A receita leva ovos e farinha',
]

demo_embeddings = embed_model.encode(demo_sentences)

# Similaridade de cosseno
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(demo_embeddings)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
im = ax1.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
short_labels = [s[:30]+'...' if len(s) > 30 else s for s in demo_sentences]
ax1.set_xticks(range(len(demo_sentences)))
ax1.set_yticks(range(len(demo_sentences)))
ax1.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax1.set_yticklabels(short_labels, fontsize=8)
ax1.set_title('Similaridade de Cosseno entre Frases', fontweight='bold')
plt.colorbar(im, ax=ax1)

# PCA 2D
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(demo_embeddings)

colors = ['#e74c3c']*4 + ['#27ae60']*3 + ['#3498db']*3
ax2.scatter(emb_2d[:, 0], emb_2d[:, 1], c=colors, s=100, zorder=5)
for i, label in enumerate(short_labels):
    ax2.annotate(label, (emb_2d[i, 0], emb_2d[i, 1]),
                fontsize=7, xytext=(5, 5), textcoords='offset points')
ax2.set_title('Embeddings em 2D (PCA)', fontweight='bold')
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)')
ax2.grid(True, alpha=0.3)

# Legenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Movimentos'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#27ae60', markersize=10, label='Estratégia'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=10, label='Irrelevante'),
]
ax2.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

print('💡 Observe: frases sobre movimentos de peças se agrupam,')
print('   frases sobre estratégia se agrupam, e temas irrelevantes ficam distantes.')

---
## 6. 🗄️ Vector Store — Busca por Similaridade

### O que é uma vector store?

Um banco de dados otimizado para buscar os **vetores mais próximos** de um vetor de query.

### Opções:

| Ferramenta | Tipo | Quando usar |
|---|---|---|
| **NumPy** (cosine sim) | Em memória | Prototipação, < 10k docs |
| **FAISS** | Em memória, otimizado | Até milhões de docs |
| **ChromaDB** | Embarcado, persistente | Aplicações locais |
| **Pinecone/Weaviate** | Serviço na nuvem | Produção, bilhões de docs |

Vamos implementar **ambos** (NumPy e FAISS) para comparar.

In [ ]:
import faiss

class SimpleVectorStore:
    """Vector store didática — implementada do zero com NumPy."""

    def __init__(self):
        self.embeddings = []   # lista de vetores
        self.chunks = []       # chunks correspondentes
        self.is_built = False

    def add(self, chunks: List[Chunk], embed_model):
        """Adiciona chunks ao store, computando embeddings."""
        texts = [c.text for c in chunks]
        embeddings = embed_model.encode(texts, show_progress_bar=True)
        self.embeddings = np.array(embeddings, dtype=np.float32)
        self.chunks = chunks
        self.is_built = True
        print(f'  Indexados {len(chunks)} chunks ({self.embeddings.shape})')

    def search(self, query: str, embed_model, top_k: int = 3) -> List[dict]:
        """Busca os top-k chunks mais similares à query."""
        query_emb = embed_model.encode([query])[0]

        # Similaridade de cosseno manual
        norms = np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_emb)
        similarities = (self.embeddings @ query_emb) / (norms + 1e-8)

        # Top-k
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                'chunk': self.chunks[idx],
                'score': float(similarities[idx]),
                'index': int(idx),
            })
        return results


class FAISSVectorStore:
    """Vector store com FAISS — mais rápida para datasets grandes."""

    def __init__(self):
        self.index = None
        self.chunks = []

    def add(self, chunks: List[Chunk], embed_model):
        texts = [c.text for c in chunks]
        embeddings = embed_model.encode(texts, show_progress_bar=True)
        embeddings = np.array(embeddings, dtype=np.float32)

        # Normalizar para usar Inner Product como Cosine Similarity
        faiss.normalize_L2(embeddings)

        # Criar índice FAISS
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)  # Inner Product (= cosine após normalização)
        self.index.add(embeddings)
        self.chunks = chunks
        self.embeddings_raw = embeddings
        print(f'  FAISS: indexados {self.index.ntotal} vetores de dim {dim}')

    def search(self, query: str, embed_model, top_k: int = 3) -> List[dict]:
        query_emb = embed_model.encode([query]).astype(np.float32)
        faiss.normalize_L2(query_emb)

        scores, indices = self.index.search(query_emb, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            results.append({
                'chunk': self.chunks[idx],
                'score': float(score),
                'index': int(idx),
            })
        return results


# Criar e popular ambos os stores
print('📦 Construindo Vector Stores...\n')

print('NumPy Store:')
numpy_store = SimpleVectorStore()
numpy_store.add(CHUNKS, embed_model)

print('\nFAISS Store:')
faiss_store = FAISSVectorStore()
faiss_store.add(CHUNKS, embed_model)

In [ ]:
# === 🔬 Experimento 3: Testar busca com diferentes queries ===

test_queries = [
    'Como o bispo se movimenta no xadrez?',
    'O que é o roque e quando pode ser feito?',
    'Quem foi o primeiro campeão mundial de xadrez?',
    'Como funciona a promoção do peão?',
    'Qual a importância do centro no xadrez?',
]

print('🔍 Teste de Retrieval:\n')
for query in test_queries:
    results = faiss_store.search(query, embed_model, top_k=2)
    print(f'❓ "{query}"')
    for r in results:
        preview = r['chunk'].text[:100].replace('\n', ' ')
        print(f'   Score={r["score"]:.3f} | Chunk {r["index"]}: "{preview}..."')
    print()

---
## 7. 🤖 Generation — Gerando Respostas com Contexto

### O prompt de RAG

O segredo do RAG é o **prompt engineering**: incluir os chunks recuperados
como contexto e instruir o modelo a basear sua resposta neles.

```
Sistema: Você é um assistente que responde baseado nos documentos fornecidos.
         Se a informação não está nos documentos, diga que não sabe.

Contexto:
[Chunk 1 recuperado]
[Chunk 2 recuperado]
[Chunk 3 recuperado]

Pergunta: Como o bispo se movimenta?
```

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Carregar modelo de geração
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'✅ Modelo de geração carregado: {MODEL_ID}')

In [ ]:
# === Prompt Templates ===

RAG_SYSTEM_PROMPT = """Você é um assistente especializado que responde perguntas APENAS com base nos documentos fornecidos no contexto.

Regras:
- Responda de forma clara e direta, citando informações do contexto.
- Se a resposta não estiver no contexto, diga: "Não encontrei essa informação nos documentos disponíveis."
- Não invente informações além do que está no contexto."""

NO_RAG_SYSTEM_PROMPT = """Você é um assistente que responde perguntas da melhor forma possível usando seu conhecimento geral."""


def generate_with_context(model, tokenizer, query, context_chunks=None,
                          max_tokens=300, temperature=0.3):
    """Gera resposta com ou sem contexto RAG."""

    if context_chunks:
        # Montar contexto
        context_text = '\n\n---\n\n'.join([
            f'[Trecho {i+1}]:\n{chunk.text}'
            for i, chunk in enumerate(context_chunks)
        ])
        user_message = f"""Contexto (documentos recuperados):

{context_text}

---

Pergunta: {query}"""
        system = RAG_SYSTEM_PROMPT
    else:
        user_message = query
        system = NO_RAG_SYSTEM_PROMPT

    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': user_message},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                skip_special_tokens=True)
    return response.strip()

print('✅ Funções de geração definidas.')

---
## 8. 🔄 Pipeline RAG Completo

Vamos montar tudo em uma classe limpa e testar.

In [ ]:
class RAGPipeline:
    """Pipeline RAG completo: query → retrieve → generate."""

    def __init__(self, vector_store, embed_model, gen_model, tokenizer, top_k=3):
        self.store = vector_store
        self.embed_model = embed_model
        self.gen_model = gen_model
        self.tokenizer = tokenizer
        self.top_k = top_k

    def query(self, question, top_k=None, show_sources=True):
        """Faz a pergunta completa: retrieve + generate."""
        k = top_k or self.top_k

        # 1. Retrieve
        results = self.store.search(question, self.embed_model, top_k=k)
        chunks = [r['chunk'] for r in results]
        scores = [r['score'] for r in results]

        # 2. Generate
        response = generate_with_context(
            self.gen_model, self.tokenizer, question, chunks
        )

        # 3. Resultado
        output = {
            'question': question,
            'answer': response,
            'sources': results,
        }

        if show_sources:
            print(f'❓ {question}\n')
            print(f'💬 {response}\n')
            print(f'📚 Fontes ({len(results)} chunks):')
            for r in results:
                preview = r['chunk'].text[:80].replace('\n', ' ')
                print(f'   Score={r["score"]:.3f} | "{preview}..."')
            print()

        return output


# Instanciar pipeline
rag = RAGPipeline(
    vector_store=faiss_store,
    embed_model=embed_model,
    gen_model=gen_model,
    tokenizer=tokenizer,
    top_k=3
)

print('✅ Pipeline RAG montado!\n')
print('=' * 60)
rag.query('Como o bispo se movimenta no xadrez?')
print('=' * 60)
rag.query('O que é o roque e quais são as condições para realizá-lo?')
print('=' * 60)
rag.query('Quem derrotou Garry Kasparov em 1997?')

---
## 9. 🔬 Experimentos

### 🔬 Experimento 4: Com RAG vs Sem RAG

A comparação mais fundamental: o RAG realmente ajuda?

In [ ]:
# === Comparação: Com RAG vs Sem RAG ===

eval_questions = [
    'Como o cavalo se movimenta no xadrez?',
    'O que é en passant?',
    'Quem foi o primeiro campeão mundial de xadrez e em que ano?',
    'O que são peões passados e por que são importantes?',
    'Qual é o valor relativo da rainha em pontos?',
    'O que é zugzwang?',
    'Quantas casas tem o tabuleiro de xadrez?',
]

results_comparison = []

print('📊 Comparação: Com RAG vs Sem RAG\n')
for q in eval_questions:
    # Sem RAG
    no_rag = generate_with_context(gen_model, tokenizer, q, context_chunks=None)

    # Com RAG
    retrieved = faiss_store.search(q, embed_model, top_k=3)
    chunks = [r['chunk'] for r in retrieved]
    with_rag = generate_with_context(gen_model, tokenizer, q, context_chunks=chunks)

    results_comparison.append({
        'question': q,
        'no_rag': no_rag,
        'with_rag': with_rag,
        'retrieval_scores': [r['score'] for r in retrieved],
    })

    print(f'❓ {q}')
    print(f'  🔴 Sem RAG: {no_rag[:200]}...' if len(no_rag) > 200 else f'  🔴 Sem RAG: {no_rag}')
    print(f'  🟢 Com RAG: {with_rag[:200]}...' if len(with_rag) > 200 else f'  🟢 Com RAG: {with_rag}')
    print(f'  📊 Scores retrieval: {[f"{s:.3f}" for s in results_comparison[-1]["retrieval_scores"]]}')
    print(f'  {"─"*60}\n')

### 🔬 Experimento 5: Efeito do Chunk Size

In [ ]:
# === Testar diferentes chunk sizes ===

chunk_sizes = [200, 300, 500, 800, 1200]
test_query = 'O que é o roque e quais são as condições para realizá-lo?'

print(f'Query: "{test_query}"\n')

chunk_size_results = []
for cs in chunk_sizes:
    chunks_test = chunk_recursive(CHESS_DOCUMENT, chunk_size=cs, overlap=cs//10)

    # Criar store temporário
    tmp_store = FAISSVectorStore()
    texts = [c.text for c in chunks_test]
    embs = embed_model.encode(texts)
    embs = np.array(embs, dtype=np.float32)
    faiss.normalize_L2(embs)
    tmp_store.index = faiss.IndexFlatIP(embs.shape[1])
    tmp_store.index.add(embs)
    tmp_store.chunks = chunks_test

    # Buscar
    results = tmp_store.search(test_query, embed_model, top_k=3)
    top_score = results[0]['score']
    avg_score = np.mean([r['score'] for r in results])

    # Gerar
    context_chunks = [r['chunk'] for r in results]
    answer = generate_with_context(gen_model, tokenizer, test_query, context_chunks)

    chunk_size_results.append({
        'chunk_size': cs,
        'n_chunks': len(chunks_test),
        'top_score': top_score,
        'avg_score': avg_score,
        'answer': answer,
    })

    print(f'  Chunk size={cs:4d} | {len(chunks_test):2d} chunks | '
          f'top_score={top_score:.3f} | avg_score={avg_score:.3f}')
    print(f'    → {answer[:120]}...')
    print()

# Plotar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sizes_p = [r['chunk_size'] for r in chunk_size_results]
top_scores_p = [r['top_score'] for r in chunk_size_results]
n_chunks_p = [r['n_chunks'] for r in chunk_size_results]

ax1.plot(sizes_p, top_scores_p, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.set_xlabel('Chunk Size (chars)')
ax1.set_ylabel('Top-1 Similarity Score')
ax1.set_title('Qualidade do Retrieval vs Chunk Size', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(sizes_p)), n_chunks_p, color='steelblue', alpha=0.7)
ax2.set_xticks(range(len(sizes_p)))
ax2.set_xticklabels([str(s) for s in sizes_p])
ax2.set_xlabel('Chunk Size (chars)')
ax2.set_ylabel('Número de Chunks')
ax2.set_title('Número de Chunks vs Chunk Size', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Efeito do Chunk Size no RAG', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Trade-off:')
print('   Chunks pequenos → mais específicos, score mais alto, mas menos contexto')
print('   Chunks grandes  → mais contexto, mas podem diluir informação relevante')
print('   Sweet spot: geralmente 300-500 caracteres')

### 🔬 Experimento 6: Efeito do Top-K

In [ ]:
# === Testar diferentes valores de top-k ===
test_query_k = 'O que é en passant e quando pode ser feito?'

top_ks = [1, 2, 3, 5, 8]

print(f'Query: "{test_query_k}"\n')

for k in top_ks:
    results = faiss_store.search(test_query_k, embed_model, top_k=k)
    chunks_k = [r['chunk'] for r in results]
    scores_k = [r['score'] for r in results]

    # Contar tokens do contexto
    context_text = ' '.join([c.text for c in chunks_k])
    n_context_tokens = len(tokenizer.encode(context_text))

    answer = generate_with_context(gen_model, tokenizer, test_query_k, chunks_k)

    print(f'  top_k={k} | scores={["{:.3f}".format(s) for s in scores_k]} | '
          f'context_tokens={n_context_tokens}')
    print(f'    → {answer[:150]}...')
    print()

print('💡 Trade-off do top-k:')
print('   k=1: resposta focada, mas pode perder informação')
print('   k=3: bom balanço (padrão)')
print('   k=5+: mais contexto, mas chunks irrelevantes podem confundir o modelo')
print('   Atenção: mais chunks = mais tokens = mais lento e caro')

---
## 10. 🚀 Técnicas Avançadas

### 🔬 Experimento 7: Relevance Threshold — Filtrando Chunks Ruins

Em vez de sempre usar top-k, podemos definir um **threshold mínimo** de similaridade.
Se nenhum chunk é relevante, melhor dizer "não sei" do que dar lixo.

In [ ]:
# === Threshold de relevância ===

# Perguntas: relevante e irrelevante
queries_threshold = [
    ('Como o bispo se move?', True),            # relevante
    ('Qual o PIB do Brasil em 2024?', False),    # irrelevante
    ('O que é xeque-mate?', True),               # relevante
    ('Quem inventou o telefone?', False),         # irrelevante
]

threshold = 0.35

print(f'Threshold de relevância: {threshold}\n')

for query, is_relevant in queries_threshold:
    results = faiss_store.search(query, embed_model, top_k=3)
    scores = [r['score'] for r in results]
    above_threshold = [r for r in results if r['score'] >= threshold]

    status = '✅' if (len(above_threshold) > 0) == is_relevant else '❌'

    if above_threshold:
        chunks = [r['chunk'] for r in above_threshold]
        answer = generate_with_context(gen_model, tokenizer, query, chunks)
    else:
        answer = 'Não encontrei informações relevantes nos documentos disponíveis.'

    print(f'{status} "{query}"')
    print(f'   Scores: {["{:.3f}".format(s) for s in scores]} | '
          f'Acima threshold: {len(above_threshold)}')
    print(f'   → {answer[:150]}')
    print()

### 🔬 Experimento 8: HyDE — Hypothetical Document Embeddings

**Ideia:** em vez de buscar diretamente pela pergunta, pedir ao LLM para **gerar
uma resposta hipotética**, e buscar por ela. A resposta hipotética estará no mesmo
"espaço semântico" dos documentos.

In [ ]:
# === HyDE: Hypothetical Document Embeddings ===

def hyde_search(query, gen_model, tokenizer, embed_model, store, top_k=3):
    """Busca via HyDE: gera documento hipotético, busca por ele."""

    # 1. Gerar resposta hipotética (sem RAG)
    hyde_prompt = f"""Escreva um parágrafo curto e informativo que responda a seguinte pergunta sobre xadrez.
Seja específico e use termos técnicos.

Pergunta: {query}"""

    hypothetical = generate_with_context(gen_model, tokenizer, hyde_prompt, context_chunks=None,
                                          max_tokens=150, temperature=0.3)

    # 2. Buscar usando o documento hipotético como query
    results = store.search(hypothetical, embed_model, top_k=top_k)

    return hypothetical, results


# Comparar busca normal vs HyDE
test_q = 'Quais os cuidados ao mover peões no xadrez?'

print(f'❓ Query: "{test_q}"\n')

# Busca normal
normal_results = faiss_store.search(test_q, embed_model, top_k=3)
print('🔍 Busca Normal:')
for r in normal_results:
    print(f'   Score={r["score"]:.3f} | "{r["chunk"].text[:80]}..."')

# HyDE
hypothetical, hyde_results = hyde_search(test_q, gen_model, tokenizer, embed_model, faiss_store)
print(f'\n🧪 HyDE:')
print(f'   Documento hipotético: "{hypothetical[:120]}..."')
for r in hyde_results:
    print(f'   Score={r["score"]:.3f} | "{r["chunk"].text[:80]}..."')

print('\n💡 HyDE pode encontrar chunks mais relevantes quando a query é vaga,')
print('   porque o documento hipotético está mais próximo da "linguagem" dos docs.')

### 🔬 Experimento 9: Avaliação Automatizada — LLM como Juiz

Uma técnica cada vez mais usada: pedir ao **próprio LLM** para avaliar a qualidade
das respostas (LLM-as-a-judge). Útil quando não temos gabaritos prontos.

In [ ]:
# === LLM-as-a-Judge ===

JUDGE_PROMPT_TEMPLATE = """Avalie a seguinte resposta numa escala de 1 a 5.

Critérios:
- 5: Resposta completa, correta e bem explicada
- 4: Resposta correta mas poderia ser mais completa
- 3: Resposta parcialmente correta
- 2: Resposta vaga ou com erros
- 1: Resposta errada ou irrelevante

Pergunta: {question}
Resposta: {answer}

Responda APENAS com o número (1-5) e uma justificativa de uma linha."""


def judge_response(question, answer, model, tokenizer):
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, answer=answer)
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.1,
                                  do_sample=True, pad_token_id=tokenizer.pad_token_id)
    verdict = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return verdict.strip()


# Avaliar respostas com e sem RAG
print('🧑‍⚖️ Avaliação Automatizada (LLM-as-a-Judge):\n')

for comp in results_comparison[:5]:
    q = comp['question']
    # Julgar sem RAG
    j_no_rag = judge_response(q, comp['no_rag'], gen_model, tokenizer)
    # Julgar com RAG
    j_with_rag = judge_response(q, comp['with_rag'], gen_model, tokenizer)

    print(f'  ❓ {q}')
    print(f'     🔴 Sem RAG:  {j_no_rag[:80]}')
    print(f'     🟢 Com RAG:  {j_with_rag[:80]}')
    print()

print('⚠️ Nota: modelos pequenos não são bons juízes.')
print('   Em produção, use um modelo grande (GPT-4, Claude) como juiz,')
print('   ou crie um gabarito humano para avaliação mais confiável.')

---
## 📝 Resumo e Referência Rápida

### Pipeline RAG

| Etapa | Componente | Decisão-chave |
|-------|-----------|---------------|
| 1. Chunking | `chunk_recursive()` | Chunk size: 300-500 chars, overlap: 10-20% |
| 2. Embedding | `all-MiniLM-L6-v2` | Modelo: depende da língua e qualidade desejada |
| 3. Indexação | FAISS / ChromaDB | Escala: FAISS para milhões, Chroma para simplicidade |
| 4. Retrieval | Cosine similarity | top-k: 3-5, com threshold de relevância |
| 5. Generation | LLM + prompt | Prompt claro com instruções de fidelidade ao contexto |

### Decisões de design

| Decisão | Recomendação | Trade-off |
|---|---|---|
| Chunk size | 300-500 chars | Menor=preciso, maior=mais contexto |
| Overlap | 10-20% do chunk | Sem overlap pode perder info na fronteira |
| Top-K | 3 (padrão) | Mais=mais contexto mas mais ruído |
| Threshold | 0.3-0.4 | Evita respostas baseadas em chunks irrelevantes |
| Embedding model | Depende da língua | Multilingual para PT-BR: e5, bge-m3 |

### Quando RAG falha

| Sintoma | Causa provável | Solução |
|---|---|---|
| Resposta errada com docs certos | LLM ignora contexto | Melhorar prompt, modelo maior |
| Docs errados recuperados | Embedding/chunking ruim | Ajustar chunk size, modelo embedding |
| "Não encontrei" para pergunta válida | Threshold alto demais | Reduzir threshold, HyDE |
| Resposta genérica | Chunks pouco específicos | Chunks menores, reranking |

### O que aprendemos

> **RAG é a forma mais prática de dar conhecimento atualizado a um LLM.**
> Não exige re-treinamento, é rastreável (você sabe de onde veio a informação),
> e escala para milhões de documentos. A qualidade depende crucialmente de
> **bom chunking** e **bom retrieval** — o LLM só é tão bom quanto o contexto que recebe.